# BERT Fine-tuning

This notebook covers BERT (Bidirectional Encoder Representations from Transformers) and how to fine-tune it for downstream NLP tasks. BERT is a revolutionary transformer-based model that has dominated NLP since its introduction in 2018.

We will explore how to fine-tune BERT for text classification using the HuggingFace Transformers library.

In [ ]:
# Import required libraries
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

print("Libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Understanding BERT

BERT is a transformer-based model that uses bidirectional attention to understand context from both directions. Unlike previous models that read text left-to-right or right-to-left, BERT considers the entire context simultaneously.

**Key innovations:**
- **Bidirectional**: Reads text in both directions simultaneously
- **Pre-training + Fine-tuning**: Pre-trained on large corpus, then fine-tuned for specific tasks
- **Masked Language Model (MLM)**: Randomly masks words and predicts them
- **Next Sentence Prediction (NSP)**: Predicts if one sentence follows another

BERT can be fine-tuned for many tasks: classification, question answering, named entity recognition, etc.

In [ ]:
# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # Binary classification
)

print("=== BERT Model Loaded ===")
print(f"Model: {model_name}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Number of labels: 2")

## 2. Preparing Data for Fine-tuning

We need to prepare our data in a format that BERT can understand. This involves tokenisation and creating PyTorch datasets.

In [ ]:
# Create a custom dataset class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenise the text
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Sample training data
train_texts = [
    "This product is amazing! I love it so much.",
    "Absolutely fantastic, highly recommend!",
    "Great quality and fast delivery.",
    "Best purchase I've ever made.",
    "Wonderful experience, will buy again.",
    "Very satisfied with this item.",
    "Excellent service and friendly staff.",
    "The food was delicious and fresh.",
    "Terrible product, complete waste of money.",
    "Very disappointed with this purchase.",
    "Worst experience ever, do not buy.",
    "Poor quality, broke after one use.",
    "Horrible customer service, never again.",
    "The product didn't work as advertised.",
    "Slow delivery and damaged package.",
    "Tasteless food, will not return.",
]

train_labels = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

# Create dataset
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)

print("=== Dataset Created ===")
print(f"Training samples: {len(train_dataset)}")

# Show a sample
sample = train_dataset[0]
print(f"\nSample item:")
print(f"  Input IDs shape: {sample['input_ids'].shape}")
print(f"  Attention mask shape: {sample['attention_mask'].shape}")
print(f"  Label: {sample['labels']}")

In [ ]:
# Test tokenisation
sample_text = "This product is amazing! I love it so much."
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("=== Tokenisation Example ===")
print(f"Original text: {sample_text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"\nBERT uses special tokens:")
print(f"  [CLS]: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"  [SEP]: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")
print(f"  [PAD]: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

## 3. Fine-tuning Configuration

We need to configure the training parameters for fine-tuning. This includes learning rate, batch size, number of epochs, etc.

In [ ]:
# Define compute metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

# Configure training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    warmup_steps=100,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

print("=== Training Configuration ===")
print(f"Output directory: {training_args.output_dir}")
print(f"Number of epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

## 4. Training the Model

Now we'll fine-tune the BERT model on our sentiment classification task.

In [ ]:
# Initialise the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)

print("=== Starting Fine-tuning ===")
print("This may take a few minutes on CPU...")

# Train the model
trainer.train()

print("\n=== Fine-tuning Complete ===")

## 5. Making Predictions

Now let's use our fine-tuned model to make predictions on new text.

In [ ]:
# Test on new sentences
test_texts = [
    "This is absolutely brilliant! Best ever!",
    "I hate this, it's the worst thing ever.",
    "The meeting starts at 2pm in conference room B.",
    "So happy with my purchase, works perfectly!",
    "Very poor quality, would not recommend.",
]

print("=== Making Predictions ===\n")

# Set model to evaluation mode
model.eval()

for text in test_texts:
    # Tokenise
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    
    # Make prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        prediction = torch.argmax(logits, dim=1).item()
        probabilities = torch.softmax(logits, dim=1)
    
    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probabilities[0][prediction].item()
    
    print(f"Text: {text}")
    print(f"Prediction: {sentiment} (confidence: {confidence:.4f})")
    print()

## 6. Using DistilBERT for Faster Inference

BERT is a large model. For faster inference, we can use DistilBERT, which is 60% faster while retaining 97% of BERT's performance.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Load DistilBERT
distil_model_name = 'distilbert-base-uncased'
distil_tokenizer = DistilBertTokenizer.from_pretrained(distil_model_name)
distil_model = DistilBertForSequenceClassification.from_pretrained(
    distil_model_name,
    num_labels=2
)

print("=== DistilBERT Model ===")
print(f"Model: {distil_model_name}")
print(f"Parameters: {sum(p.numel() for p in distil_model.parameters()):,}")
print(f"\nComparison:")
print(f"  BERT-base: ~110M parameters")
print(f"  DistilBERT: ~66M parameters (40% smaller)")

## Summary

In this notebook, we covered BERT fine-tuning:

1. **Understanding BERT**: Learned about BERT's bidirectional architecture and pre-training
2. **Loading Models**: Used HuggingFace to load pre-trained BERT
3. **Data Preparation**: Created custom PyTorch datasets for BERT
4. **Tokenisation**: Explored BERT's tokenisation process
5. **Fine-tuning**: Fine-tuned BERT for sentiment classification
6. **Predictions**: Made predictions on new text
7. **DistilBERT**: Explored lighter alternatives for faster inference

BERT and its variants are the foundation of modern NLP. Fine-tuning allows you to leverage pre-trained knowledge for your specific tasks with relatively small amounts of task-specific data.